# 🧠 Magic Survival AI — Kaggle Training Notebook (Skill Selection + Computer Vision)

Notebook này huấn luyện **2 hệ thống AI** từ bộ dữ liệu `kaggle_magic_survival_dataset.zip` được xuất từ máy của bạn:
1. **Skill Value & Preference Network (`skill_value_net.json`)**:
   - Mạng Neural MLP học dự đoán **Giá trị sinh tồn kỳ vọng $Q(s, a)$** và **Học bắt chước lựa chọn của Người chơi (Pairwise Margin Ranking Loss)**.
   - Xuất trọng số trực tiếp ra `skill_value_net.json` để bot trên máy cá nhân chạy bằng `numpy` trong `< 0.1ms` mà **không cần cài PyTorch trên máy cá nhân**.
2. **Computer Vision CNNs (`card_vision.onnx` & `screen_vision.onnx`)**:
   - **ScreenVisionCNN**: Phân loại màn hình game (`level_up`, `chest`, `attribute`, `strengthen_magic`, `endgame`, ...).
   - **CardVisionCNN**: Nhận diện trực tiếp kỹ năng trên từng ô thẻ bài (`192x64`) từ hình ảnh không cần OCR.
   - Xuất ra chuẩn **ONNX** để bot trên máy cá nhân chạy trực tiếp bằng `cv2.dnn.readNetFromONNX` có sẵn trong OpenCV.

In [ ]:
import json
import math
import os
from pathlib import Path
import random
import shutil
import zipfile

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, random_split

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', DEVICE)

# Tự động tìm đường dẫn Dataset trên Kaggle (/kaggle/input/...) hoặc trên máy local
CANDIDATE_DIRS = [
    Path('/kaggle/input'),
    Path('data/kaggle_export'),
    Path('../data/kaggle_export'),
]

DATA_ROOT = None
for base in CANDIDATE_DIRS:
    if not base.exists():
        continue
    if (base / 'vocab.json').is_file():
        DATA_ROOT = base
        break
    found = list(base.rglob('vocab.json'))
    if found:
        DATA_ROOT = found[0].parent
        break

assert DATA_ROOT is not None, 'Không tìm thấy vocab.json! Hãy Add Input dataset kaggle_magic_survival_dataset.zip vào Kaggle Notebook.'
OUTPUT_DIR = Path('/kaggle/working/ai_models') if Path('/kaggle/working').exists() else Path('ai_models')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

vocab_meta = json.loads((DATA_ROOT / 'vocab.json').read_text(encoding='utf-8'))
VOCAB = vocab_meta['vocab']
SKILL_TO_IDX = {s: i for i, s in enumerate(VOCAB)}
FUSION_PARTNERS = {k: set(v) for k, v in vocab_meta.get('fusion_partners', {}).items()}
print(f'Loaded dataset from: {DATA_ROOT}')
print(f'Total canonical skills: {len(VOCAB)} | Decisions: {vocab_meta.get("total_skill_decisions")} | Card crops: {vocab_meta.get("total_card_images")}')

## Phần 1: Huấn luyện AI Chọn Phép (`SkillValueMLP`)
Kết hợp **Hồi quy thời gian sinh tồn (Value Regression)** và **Xếp hạng cặp thẻ (Pairwise Ranking Loss: Thẻ được chọn > Thẻ bị bỏ qua trên cùng màn hình)**.

In [ ]:
def build_feature_vec(candidate_skill, active_skills, skill_levels, step, offered_level=None, has_asterisk=False):
    v_len = len(VOCAB)
    feat = np.zeros(2 * v_len + 6, dtype=np.float32)
    active_set = set(active_skills)
    for sk in active_set:
        idx = SKILL_TO_IDX.get(sk)
        if idx is not None:
            lvl = skill_levels.get(sk, 2)
            feat[idx] = min(1.0, float(lvl) / 7.0)
    c_idx = SKILL_TO_IDX.get(candidate_skill)
    if c_idx is not None:
        feat[v_len + c_idx] = 1.0
    curr_lvl = skill_levels.get(candidate_skill, 1 if candidate_skill in active_set else 0)
    eff_off = offered_level if offered_level is not None else (curr_lvl + 1)
    partners = FUSION_PARTNERS.get(candidate_skill, set())
    has_partner = 1.0 if (partners and (partners & active_set)) else 0.0
    base = 2 * v_len
    feat[base + 0] = min(1.0, max(0.0, float(step) / 100.0))
    feat[base + 1] = min(1.0, float(len(active_set)) / 15.0)
    feat[base + 2] = min(1.0, float(curr_lvl) / 7.0)
    feat[base + 3] = 1.0 if eff_off == 7 else 0.0
    feat[base + 4] = 1.0 if has_asterisk else 0.0
    feat[base + 5] = has_partner
    return feat

df_dec = pd.read_csv(DATA_ROOT / 'skill_decisions.csv')
print(f'Loaded {len(df_dec)} skill decisions.')

pos_features = []
neg_features = []
has_neg_mask = []
target_values = []
sample_weights = []

median_surv = max(300.0, float(df_dec['survival_sec'].median()) if len(df_dec) else 500.0)
map_medians = {}
if 'game_map' in df_dec.columns:
    for m_name, grp in df_dec.groupby(df_dec['game_map'].fillna('').astype(str).str.lower().str.strip()):
        if m_name and len(grp) >= 5:
            map_medians[m_name] = max(240.0, float(grp['survival_sec'].median()))
print('Per-map survival medians:', map_medians or {'global': median_surv})

for _, row in df_dec.iterrows():
    chosen = str(row['chosen_skill'])
    if chosen not in SKILL_TO_IDX:
        continue
    active = json.loads(row['active_skills_json'])
    levels = json.loads(row['skill_levels_json'])
    rejected = json.loads(row['rejected_skills_json'])
    step = int(row['step'])
    off_lvl = int(row['offered_level']) if not pd.isna(row['offered_level']) else None
    surv = float(row['survival_sec'])
    is_hum = int(row['is_human']) == 1
    row_map = str(row['game_map']).lower().strip() if 'game_map' in df_dec.columns and not pd.isna(row['game_map']) else ''
    row_median = map_medians.get(row_map, median_surv)

    # Điểm mục tiêu chuẩn hóa công bằng theo từng bản đồ trong khoảng [-0.8, +1.5]
    norm_val = np.clip((surv - row_median) / max(240.0, row_median * 0.75), -0.8, 1.5)
    if is_hum:
        norm_val = max(0.8, norm_val + 0.5)

    f_pos = build_feature_vec(chosen, active, levels, step, offered_level=off_lvl)
    if rejected:
        for rej_sk in rejected:
            if rej_sk in SKILL_TO_IDX:
                f_neg = build_feature_vec(rej_sk, active, levels, step)
                pos_features.append(f_pos)
                neg_features.append(f_neg)
                has_neg_mask.append(1.0)
                target_values.append(norm_val)
                sample_weights.append(2.0 if is_hum else 1.0)
    else:
        # Tạo mẫu đối chứng ngẫu nhiên từ một kỹ năng ngoài bộ đang có
        rand_sk = random.choice(VOCAB)
        f_neg = build_feature_vec(rand_sk, active, levels, step)
        pos_features.append(f_pos)
        neg_features.append(f_neg)
        has_neg_mask.append(0.3 if norm_val > 0 else 0.0)
        target_values.append(norm_val)
        sample_weights.append(1.0)

X_pos = torch.tensor(np.array(pos_features), dtype=torch.float32, device=DEVICE)
X_neg = torch.tensor(np.array(neg_features), dtype=torch.float32, device=DEVICE)
M_neg = torch.tensor(np.array(has_neg_mask), dtype=torch.float32, device=DEVICE)
Y_val = torch.tensor(np.array(target_values), dtype=torch.float32, device=DEVICE)
W_smp = torch.tensor(np.array(sample_weights), dtype=torch.float32, device=DEVICE)

class SkillValueMLP(nn.Module):
    def __init__(self, in_dim: int):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x).squeeze(-1)

in_dim = X_pos.shape[1]
skill_model = SkillValueMLP(in_dim).to(DEVICE)
optimizer = torch.optim.AdamW(skill_model.parameters(), lr=2e-3, weight_decay=1e-4)

EPOCHS = 80
for epoch in range(1, EPOCHS + 1):
    skill_model.train()
    optimizer.zero_grad()
    q_pos = skill_model(X_pos)
    q_neg = skill_model(X_neg)

    # 1. Value regression loss
    reg_loss = (W_smp * F.smooth_l1_loss(q_pos, Y_val, reduction='none')).mean()
    # 2. Pairwise ranking loss: q_pos phải lớn hơn q_neg ít nhất 0.35 điểm
    rank_loss = (M_neg * W_smp * F.relu(0.35 - (q_pos - q_neg))).mean()
    loss = reg_loss + 1.5 * rank_loss
    loss.backward()
    optimizer.step()

    if epoch % 20 == 0 or epoch == 1:
        with torch.no_grad():
            pair_acc = ((q_pos > q_neg).float() * (M_neg > 0.5).float()).sum() / max(1.0, (M_neg > 0.5).float().sum())
        print(f'Epoch {epoch:02d}/{EPOCHS} | Loss: {loss.item():.4f} (Reg: {reg_loss.item():.4f}, Rank: {rank_loss.item():.4f}) | Pairwise Acc: {pair_acc.item()*100:.1f}%')

# Xuất trọng số ra JSON thuần để bot chạy bằng NumPy không cần cài PyTorch
skill_model.eval()
exported_net = {
    'version': 1,
    'vocab': VOCAB,
    'input_dim': in_dim,
    'layers': [
        {'weight': skill_model.fc1.weight.detach().cpu().numpy().round(6).tolist(), 'bias': skill_model.fc1.bias.detach().cpu().numpy().round(6).tolist()},
        {'weight': skill_model.fc2.weight.detach().cpu().numpy().round(6).tolist(), 'bias': skill_model.fc2.bias.detach().cpu().numpy().round(6).tolist()},
        {'weight': skill_model.fc3.weight.detach().cpu().numpy().round(6).tolist(), 'bias': skill_model.fc3.bias.detach().cpu().numpy().round(6).tolist()},
    ],
}
json_out = OUTPUT_DIR / 'skill_value_net.json'
json_out.write_text(json.dumps(exported_net), encoding='utf-8')
print(f'[OK] Đã xuất Skill Value Network: {json_out} ({json_out.stat().st_size / 1024:.1f} KB)')

## Phần 2: Huấn luyện AI Thị Giác (`CardVisionCNN` & `ScreenVisionCNN`) và xuất ONNX
Huấn luyện mạng CNN nhận diện kỹ năng trên thẻ bài (`192x64`) và phân loại màn hình (`96x160`), sau đó xuất ra `.onnx` để OpenCV (`cv2.dnn`) đọc trực tiếp.

In [ ]:
class ImageCSVDataset(Dataset):
    def __init__(self, csv_path: Path, root_dir: Path, classes: list[str], size_wh: tuple[int, int], augment: bool = False):
        self.df = pd.read_csv(csv_path)
        self.root_dir = root_dir
        self.classes = classes
        self.cls_to_idx = {c: i for i, c in enumerate(classes)}
        self.df = self.df[self.df['label'].isin(self.cls_to_idx)].reset_index(drop=True)
        self.size_wh = size_wh
        self.augment = augment
        # Đọc song song 16 luồng toàn bộ ảnh vào RAM để không bị nghẽn ổ đĩa FUSE /kaggle/input
        from concurrent.futures import ThreadPoolExecutor
        self.cached_imgs = {}
        paths = list(enumerate(self.df['image_path'].tolist()))
        def _load_one(item):
            i, rel = item
            raw = cv2.imread(str(self.root_dir / rel))
            if raw is None:
                return i, np.zeros((self.size_wh[1], self.size_wh[0], 3), dtype=np.uint8)
            return i, cv2.cvtColor(cv2.resize(raw, self.size_wh), cv2.COLOR_BGR2RGB)
        with ThreadPoolExecutor(max_workers=16) as pool:
            for i, arr in pool.map(_load_one, paths):
                self.cached_imgs[i] = arr

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = self.cached_imgs.get(idx)
        if img is None:
            img_path = self.root_dir / row['image_path']
            raw = cv2.imread(str(img_path))
            if raw is None:
                img = np.zeros((self.size_wh[1], self.size_wh[0], 3), dtype=np.uint8)
            else:
                raw = cv2.resize(raw, self.size_wh)
                img = cv2.cvtColor(raw, cv2.COLOR_BGR2RGB)
            self.cached_imgs[idx] = img
        x = img.astype(np.float32) / 255.0
        if self.augment:
            # Biến thiên độ sáng & nhiễu nhẹ giúp chống lóa hiệu ứng phép
            gain = random.uniform(0.85, 1.15)
            bias = random.uniform(-0.05, 0.05)
            x = np.clip(x * gain + bias, 0.0, 1.0)
        x = np.transpose(x, (2, 0, 1))  # (C, H, W)
        y = self.cls_to_idx[row['label']]
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

class TinyVisionCNN(nn.Module):
    def __init__(self, num_classes: int):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 24, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(24),
            nn.ReLU(inplace=True),
            nn.Conv2d(24, 48, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(48),
            nn.ReLU(inplace=True),
            nn.Conv2d(48, 96, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(96),
            nn.ReLU(inplace=True),
            nn.Conv2d(96, 128, kernel_size=3, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

def export_fused_cnn_npz(model: nn.Module, npz_path: Path):
    """Gộp trọng số Conv2d + BatchNorm2d và xuất ra file .npz nhẹ để chạy trực tiếp bằng NumPy (không cần onnxscript)."""
    model.eval().cpu()
    weights = {}
    for block_idx, conv_idx in enumerate((0, 3, 6, 9)):
        conv = model.features[conv_idx]
        bn = model.features[conv_idx + 1]
        fused = torch.nn.utils.fusion.fuse_conv_bn_eval(conv, bn)
        weights[f'w{block_idx}'] = fused.weight.detach().numpy().astype(np.float32)
        weights[f'b{block_idx}'] = fused.bias.detach().numpy().astype(np.float32)
    weights['fc_w'] = model.classifier.weight.detach().numpy().astype(np.float32)
    weights['fc_b'] = model.classifier.bias.detach().numpy().astype(np.float32)
    np.savez_compressed(str(npz_path), **weights)
    print(f'[OK] Đã xuất NumPy Vision CNN: {npz_path} ({npz_path.stat().st_size / 1024:.1f} KB)')

def train_and_export_cnn(name: str, csv_file: Path, classes: list[str], size_wh: tuple[int, int], npz_path: Path, epochs: int = 15):
    full_ds = ImageCSVDataset(csv_file, DATA_ROOT, classes, size_wh=size_wh, augment=True)
    if len(full_ds) < 10 or len(classes) < 2:
        print(f'[SKIP] {name}: Không đủ dữ liệu ({len(full_ds)} mẫu).')
        return
    val_size = max(1, int(len(full_ds) * 0.15))
    train_size = len(full_ds) - val_size
    train_ds, val_ds = random_split(full_ds, [train_size, val_size])
    train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

    model = TinyVisionCNN(len(classes)).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)

    for ep in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for bx, by in train_loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad()
            out = model(bx)
            loss = F.cross_entropy(out, by)
            loss.backward()
            opt.step()
            total_loss += loss.item() * bx.size(0)
        sched.step()
        if ep % 5 == 0 or ep == epochs:
            model.eval()
            correct, total = 0, 0
            with torch.no_grad():
                for vx, vy in val_loader:
                    vx, vy = vx.to(DEVICE), vy.to(DEVICE)
                    preds = model(vx).argmax(dim=1)
                    correct += (preds == vy).sum().item()
                    total += vy.size(0)
            print(f'[{name}] Epoch {ep:02d}/{epochs} | Train Loss: {total_loss/train_size:.4f} | Val Acc: {correct/max(1,total)*100:.1f}%')

    export_fused_cnn_npz(model, npz_path)

# Lọc các kỹ năng có ít nhất 5 ảnh thẻ bài để train CardVisionCNN
card_counts = vocab_meta.get('card_class_counts', {})
valid_card_classes = sorted([k for k, cnt in card_counts.items() if cnt >= 5])
valid_screen_classes = sorted(vocab_meta.get('screen_classes', []))

card_size_wh = tuple(vocab_meta.get('card_size_wh', [192, 64]))
screen_size_wh = tuple(vocab_meta.get('screen_size_wh', [96, 160]))

train_and_export_cnn('CardVisionCNN', DATA_ROOT / 'vision_cards.csv', valid_card_classes, card_size_wh, OUTPUT_DIR / 'card_vision.npz', epochs=15)
train_and_export_cnn('ScreenVisionCNN', DATA_ROOT / 'vision_screens.csv', valid_screen_classes, screen_size_wh, OUTPUT_DIR / 'screen_vision.npz', epochs=12)

vision_meta = {
    'card_classes': valid_card_classes,
    'screen_classes': valid_screen_classes,
    'card_size_wh': list(card_size_wh),
    'screen_size_wh': list(screen_size_wh),
}
(OUTPUT_DIR / 'vision_meta.json').write_text(json.dumps(vision_meta, indent=2, ensure_ascii=False), encoding='utf-8')

# Đóng gói toàn bộ model đã train thành ai_models.zip để tải 1 phát về máy
zip_out = OUTPUT_DIR.parent / 'ai_models.zip'
with zipfile.ZipFile(zip_out, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for f in OUTPUT_DIR.glob('*'):
        if f.is_file():
            zf.write(f, f.name)
print(f'\n🎉 HOÀN TẤT! Tải file {zip_out} về giải nén vào thư mục ai_models/ của project.')